# Домашка
1. Пофиксить loss nan (поиграться с фичами в CustomDataset)
2. Добавить в run_train проверку на valid

In [1]:
!pip install torchmetrics tensorboard --quiet
%load_ext tensorboard
!rm -rf runs

In [2]:
import pickle

import numpy as np
import pandas as pd

import torch
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader

from torchmetrics import Accuracy

In [3]:
class CustomDataset(Dataset):
    def __init__(self, dataset_path: str) -> None:
        data = pd.read_parquet(dataset_path)

        self.nrof_emb_categories = {
            "workclass_cat": 9,
            "education_cat": 16,
            "marital-status_cat": 7,
            "occupation_cat": 16,
            "relationship_cat": 6,
            "race_cat": 5,
            "sex_cat": 2,
            "native-country_cat": 42,
        }
        self.embedding_columns = list(self.nrof_emb_categories.keys())
        self.numeric_columns = [
            "age",
            "fnlwgt",
            "education-num",
            "capital-gain",
            "capital-loss",
            "hours-per-week",
        ]
        self.columns = self.embedding_columns + self.numeric_columns
        self.X = data[self.columns].reset_index(drop=True)

        for col in self.numeric_columns:
            if self.X[col].isna().any():
                self.X[col] = self.X[col].fillna(self.X[col].median())

        for col in ["fnlwgt", "capital-gain", "capital-loss"]:
            self.X[col] = np.log1p(self.X[col].astype(np.float64))

        self.y = np.asarray([0 if el == "<50k" else 1 for el in data["salary"].values], dtype=np.int32)

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, idx) -> tuple:
        row = self.X.iloc[idx]
        features = {col: torch.tensor(row[col], dtype=torch.float32) for col in self.columns}
        label = torch.tensor(self.y[idx], dtype=torch.float32)
        return features, label

In [4]:
class DenseFeatureLayer(torch.nn.Module):
    def __init__(
        self,
        nrof_cat: int,
        emb_dim: int,
        emb_columns: list[str],
        numeric_columns: list[str],
    ) -> None:
        super().__init__()

        self.emb_columns = emb_columns
        self.numeric_columns = numeric_columns
        self.numeric_feature_bn = torch.nn.BatchNorm1d(len(numeric_columns), eps=1e-3)

        input_size = len(emb_columns) + len(numeric_columns)
        self.first_feature_bn = torch.nn.BatchNorm1d(input_size, eps=1e-3)
        self.second_feature_bn = torch.nn.BatchNorm1d(input_size, eps=1e-3)

        # first order feature interactions
        self.first_order_embd = torch.nn.ModuleDict()
        for col in self.emb_columns:
            self.first_order_embd[col] = torch.nn.Embedding(nrof_cat[col], 1)
        self.first_order_scalar = torch.nn.ParameterDict({})
        for col in numeric_columns:
            self.first_order_scalar[col] = torch.nn.Parameter(torch.nn.init.xavier_uniform_(torch.empty(1, 1)))

        # second order feature interactions
        self.second_order_embd = torch.nn.ModuleDict({})
        for col in self.emb_columns:
            self.second_order_embd[col] = torch.nn.Embedding(nrof_cat[col], emb_dim)
        self.second_order_scalar = torch.nn.ParameterDict({})
        for col in numeric_columns:
            self.second_order_scalar[col] = torch.nn.Parameter(torch.nn.init.xavier_uniform_(torch.empty(emb_dim, 1)))

    def forward(self, input_data):
        numeric_features = torch.stack([input_data[col].squeeze(-1) for col in self.numeric_columns], dim=1)
        numeric_features = self.numeric_feature_bn(numeric_features)

        # first order feature interactions
        first_order_parts = []
        # categorical_columns
        for col in self.emb_columns:
            idx = input_data[col].long().squeeze(-1)
            first_order_parts.append(self.first_order_embd[col](idx))
        # numeric_columns
        for i, col in enumerate(self.numeric_columns):
            first_order_parts.append(
                torch.mul(
                    numeric_features[:, i],
                    self.first_order_scalar[col].squeeze(-1),
                ).unsqueeze(1)
            )
        first_order_embd_output = torch.cat(first_order_parts, dim=1)

        # second order feature interactions
        second_order_parts = []
        # categorical_columns
        for col in self.emb_columns:
            idx = input_data[col].long().squeeze(-1)
            second_order_parts.append(
                self.second_order_embd[col](idx).unsqueeze(1)
            )
        # numeric_columns
        for i, col in enumerate(self.numeric_columns):
            scalar = self.second_order_scalar[col].squeeze(-1)
            col_feature = numeric_features[:, i].unsqueeze(1).unsqueeze(2)
            second_order_parts.append(
                col_feature * scalar.unsqueeze(0).unsqueeze(0)
            )
        second_order_embd_output = torch.cat(second_order_parts, dim=1)

        first_order_embd_output = self.first_feature_bn(first_order_embd_output)
        second_order_embd_output = self.second_feature_bn(second_order_embd_output)

        return first_order_embd_output, second_order_embd_output

In [5]:
class FMLayer(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, first_order_embd, second_order_embd):
        # sum_square part
        summed_features_embd = torch.sum(second_order_embd, dim=1)
        summed_features_embd_square = torch.square(summed_features_embd)

        # square_sum part
        squared_features_embd = torch.square(second_order_embd)
        squared_sum_features_embd = torch.sum(squared_features_embd, dim=1)

        # second order
        second_order = 0.5 * torch.sub(summed_features_embd_square, squared_sum_features_embd)

        return first_order_embd, second_order

In [6]:
class MLPLayer(torch.nn.Module):
    def __init__(
        self,
        input_size: int,
        nrof_layers: int,
        nrof_neurons: int,
        output_size: int,
    ) -> None:
        super().__init__()

        all_input_sizes = [input_size] + [nrof_neurons] * (nrof_layers - 1)
        layers = []
        for i in range(nrof_layers - 1):
            linear = torch.nn.Linear(all_input_sizes[i], nrof_neurons)
            torch.nn.init.xavier_uniform_(linear.weight, gain=torch.nn.init.calculate_gain("relu"))
            torch.nn.init.zeros_(linear.bias)
            layers.extend([
                linear,
                torch.nn.BatchNorm1d(nrof_neurons),
                torch.nn.ReLU(),
            ])

        self.deep_block = torch.nn.Sequential(*layers)
        self.output_layer = torch.nn.Linear(nrof_neurons, output_size)
        torch.nn.init.xavier_uniform_(self.output_layer.weight)
        torch.nn.init.zeros_(self.output_layer.bias)

    def forward(self, input_data):
        output = self.deep_block(input_data)
        output = self.output_layer(output)
        return output

In [7]:
class DeepFMNet(torch.nn.Module):
    def __init__(
        self,
        nrof_cat: int,
        emb_dim: int,
        emb_columns: list[str],
        numeric_columns: list[str],
        nrof_layers: int,
        nrof_neurons: int,
        output_size: int,
        nrof_out_classes: int,
    ) -> None:
        super().__init__()

        self.emb_dim = emb_dim
        self.emb_columns = emb_columns
        self.numeric_columns = numeric_columns

        self.features_embd = DenseFeatureLayer(nrof_cat, emb_dim, emb_columns, numeric_columns)
        self.FM = FMLayer()

        input_size = (len(emb_columns) + len(numeric_columns)) * emb_dim
        self.MLP = MLPLayer(input_size, nrof_layers, nrof_neurons, output_size)

        input_size = len(emb_columns) + len(numeric_columns) + emb_dim + output_size
        self.dense_layer = torch.nn.Linear(input_size, nrof_out_classes)

    def forward(self, input_data):
        first_order_embd, second_order_embd = self.features_embd(input_data)

        FM_first_order, FM_second_order = self.FM(first_order_embd, second_order_embd)

        second_order_embd = torch.reshape(second_order_embd, [-1, (len(self.emb_columns) + len(self.numeric_columns)) * self.emb_dim])

        Deep = self.MLP(second_order_embd)

        concat_output = torch.cat([FM_first_order, FM_second_order, Deep], dim=1)

        output = self.dense_layer(concat_output)
        output = torch.squeeze(output, 1)

        return output

In [8]:
EPOCHS = 50
EMBEDDING_SIZE = 5
BATCH_SIZE = 512
NROF_LAYERS = 3
NROF_NEURONS = 50
DEEP_OUTPUT_SIZE = 50
NROF_OUT_CLASSES = 1
LEARNING_RATE = 3e-4
TRAIN_PATH = "train_adult.parquet"
VALID_PATH = "valid_adult.parquet"

In [9]:
class DeepFM:
    def __init__(self) -> None:
        self.train_dataset = CustomDataset(TRAIN_PATH)
        self.train_loader = DataLoader(
            dataset=self.train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
        )

        valid_dataset = CustomDataset(VALID_PATH)
        self.valid_loader = DataLoader(dataset=valid_dataset, batch_size=BATCH_SIZE)

        self.build_model()
        self.train_writer = SummaryWriter("runs/train")
        self.valid_writer = SummaryWriter("runs/valid")

    def build_model(self) -> None:
        self.network = DeepFMNet(
            nrof_cat=self.train_dataset.nrof_emb_categories,
            emb_dim=EMBEDDING_SIZE,
            emb_columns=self.train_dataset.embedding_columns,
            numeric_columns=self.train_dataset.numeric_columns,
            nrof_layers=NROF_LAYERS, nrof_neurons=NROF_NEURONS,
            output_size=DEEP_OUTPUT_SIZE,
            nrof_out_classes=NROF_OUT_CLASSES,
        )
        self.loss = torch.nn.BCEWithLogitsLoss()
        self.train_accuracy = Accuracy(task="binary")
        self.valid_accuracy = Accuracy(task="binary")
        self.optimizer = optim.Adam(self.network.parameters(), lr=LEARNING_RATE)

    def run_train(self) -> None:
        step = 0
        for epoch in range(1, EPOCHS + 1):
            self.network.train()
            self.train_accuracy.reset()
            for features, label in self.train_loader:
                # Reset gradients
                self.optimizer.zero_grad()
                output = self.network(features)

                # Calculate error and backpropagate
                loss = self.loss(output, label)
                output = torch.sigmoid(output)
                loss.backward()
                acc = self.train_accuracy(output, label).item()

                # Update weights with gradients
                self.optimizer.step()

                self.train_writer.add_scalar("CrossEntropyLoss", loss, step)

                step += 1
            self.train_writer.add_scalar("Accuracy", self.train_accuracy.compute().item(), step)


            self.network.eval()
            self.valid_accuracy.reset()
            total_loss = 0.0
            n = 0
            with torch.no_grad():
                for features, label in self.valid_loader:
                    output = self.network(features)
                    loss = self.loss(output, label)
                    output = torch.sigmoid(output)

                    self.valid_accuracy(output, label).item()
                    total_loss += loss.item()
                    n += 1

            self.valid_writer.add_scalar("CrossEntropyLoss", total_loss / n, step)
            self.valid_writer.add_scalar("Accuracy", self.valid_accuracy.compute().item(), step)

In [10]:
deep_fm = DeepFM()
deep_fm.run_train()

In [12]:
%tensorboard --logdir runs